# 06 — Serialization, ONNX, and Explainability

Goal: persist models safely and explain predictions using SHAP.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'pipeline').exists():
    for parent in PROJECT_ROOT.parents:
        if (parent / 'pipeline').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

Project root: /home/ahmad/AI/Github/40 AI-ML Projects for Beginners/MLOps, UI, and Deployment/Deploy a Machine Learning Model with Docker


## 6.1 Serialization Tradeoffs: Pickle vs Joblib vs ONNX
**Definition:** Serialization stores trained model state for later inference.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** Joblib is practical for sklearn/numpy artifacts; ONNX improves cross-runtime portability.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Version metadata and keep runtime compatibility matrix.

**Common Mistakes:** Loading untrusted pickle files creates security risk.

In [2]:
from pathlib import Path
import json
import joblib
import numpy as np

models = Path('models')
meta = json.loads((models / 'model_metadata.json').read_text())
model = joblib.load(models / 'best_model.joblib')

sample = np.array([[8.3,42,6.9,1.0,230,3.2,37.9,-122.2]], dtype=np.float32)
print('Best model:', meta['best_model'])
print('Prediction:', float(model.predict(sample)[0]))
print('ONNX exists:', (models / 'model_demo.onnx').exists())

Best model: LightGBM
Prediction: 3.986889817334361
ONNX exists: True


## 6.2 Explainable AI with SHAP
**Definition:** SHAP attributes prediction contributions to features in additive form.

**Theory:** Explain mechanism, assumptions, and where this fits in deployment lifecycle.

**Motivation:** Why this matters for reliability, reproducibility, and operations.

**Real-World Example:** High MedInc contribution can justify higher predicted house value in one sample.

**Visual Explanation:** See figure/code cell below.

**Code Explanation:** Code cell demonstrates concrete implementation details.

**Best Practices:** Use local and global explanations together.

**Common Mistakes:** Confusing SHAP contribution sign with causal effect.

In [3]:
from app.model import explain

features = [8.3,42,6.9,1.0,230,3.2,37.9,-122.2]
exp = explain(features)
print('Base value:', round(exp['base_value'], 4))
print('Prediction:', round(exp['prediction'], 4))
print('Top 3 contributors:')
for name, value in sorted(exp['shap_values'].items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]:
    print(f'  {name}: {value:.4f}')

Base value: 2.0719
Prediction: 3.9869
Top 3 contributors:
  MedInc: 1.9900
  Latitude: -0.4231
  Longitude: 0.3264
